# YOLO11n-Seg LKA → SimAM Head Strong — Mr_TU Data, Grouped Near-Duplicate Split

This notebook trains the indicated configuration on `Mr_TU_ShrimpDiseaseSeg_Dat-Copy` version 1. It creates or reuses a derived grouped near-duplicate split before training.

Prepared dataset: `mrtu_yolo26_grouped_near_duplicate_split`. Run cells from top to bottom.


# YOLO11n-seg Nhom C - LKA Head

Architecture experiment for YOLO11n-seg shrimp disease segmentation. Run cells top to bottom. The YAML uses actual YOLO11n channels after width scaling to avoid custom-module channel mismatches.


In [ ]:
# Install dependencies
import importlib.util
import subprocess
import sys


def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])


ensure_package("roboflow")
from importlib.metadata import PackageNotFoundError, version

PINNED_ULTRALYTICS_VERSION = "8.4.61"
try:
    installed_ultralytics = version("ultralytics")
except PackageNotFoundError:
    installed_ultralytics = None
if installed_ultralytics != PINNED_ULTRALYTICS_VERSION:
    if any(name == "ultralytics" or name.startswith("ultralytics.") for name in sys.modules):
        raise RuntimeError("Restart the kernel before pinning Ultralytics for this notebook.")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        f"ultralytics=={PINNED_ULTRALYTICS_VERSION}",
    ])
if version("ultralytics") != PINNED_ULTRALYTICS_VERSION:
    raise RuntimeError("Could not pin the required Ultralytics version.")
ensure_package("yaml", "pyyaml")


In [ ]:
# GPU check
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))


In [ ]:
# Download Mr_TU_ShrimpDiseaseSeg_Dat-Copy v1 as a YOLO26 segmentation export.
# Store the Roboflow key in a Kaggle Secret named ROBOFLOW_API_KEY (or in the
# environment); never paste a real key into this notebook.
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

WORK_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else (
    Path('/content') if Path('/content').exists() else Path.cwd()
)
ROBOFLOW_WORKSPACE = 'lets-try-this'
# Roboflow API requires the project slug, not its display name.
ROBOFLOW_PROJECT = 'mr_tu_shrimpdiseaseseg_dat-copy'
EXPECTED_PROJECT_ID = 'lets-try-this/mr_tu_shrimpdiseaseseg_dat-copy'
EXPECTED_PROJECT_TYPE = 'instance-segmentation'
EXPECTED_IMAGE_COUNT = 223
EXPECTED_CLASS_NAMES = {'BG', 'WSSV'}
ROBOFLOW_VERSION = 1
ROBOFLOW_FORMAT = 'yolo26'

if importlib.util.find_spec('roboflow') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'roboflow'])

from roboflow import Roboflow

def get_roboflow_api_key():
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
        if value:
            return value.strip()
    except Exception:
        pass
    return os.environ.get('ROBOFLOW_API_KEY', '').strip()

api_key = get_roboflow_api_key()
if not api_key:
    raise RuntimeError(
        'Missing ROBOFLOW_API_KEY. Add it to Kaggle Secrets or the environment; do not paste it into the notebook.'
    )

rf = Roboflow(api_key=api_key)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
if project.id != EXPECTED_PROJECT_ID:
    raise RuntimeError(f'Wrong Roboflow project loaded: expected={EXPECTED_PROJECT_ID}, got={project.id}')
if project.type != EXPECTED_PROJECT_TYPE:
    raise RuntimeError(f'Wrong project type: expected={EXPECTED_PROJECT_TYPE}, got={project.type}')
if project.images != EXPECTED_IMAGE_COUNT:
    raise RuntimeError(f'Wrong project image count: expected={EXPECTED_IMAGE_COUNT}, got={project.images}')
if set(project.classes) != EXPECTED_CLASS_NAMES:
    raise RuntimeError(f'Wrong project classes: expected={EXPECTED_CLASS_NAMES}, got={set(project.classes)}')

dataset = project.version(ROBOFLOW_VERSION).download(ROBOFLOW_FORMAT)
DATASET_LOCATION = Path(dataset.location)
RAW_YOLO_DATASET_PATH = DATASET_LOCATION
print('Downloaded dataset location:', DATASET_LOCATION)


In [ ]:
# Leakage-aware derived split for Mr_TU_ShrimpDiseaseSeg_Dat-Copy.
#
# This cell never moves files out of the Roboflow export. It materializes a
# derived dataset using hard links where possible (copies otherwise), then
# writes a manifest and audit files alongside its data.yaml. Its grouping is:
#   1) filename convention group: disease::shrimp_id, when available;
#   2) verified visual near-duplicate cluster, including duplicates whose
#      filenames do not match the convention;
#   3) a singleton for everything else.
# Every connected group is assigned to exactly one split.

import csv
import hashlib
import importlib.util
import json
import os
import random
import re
import shutil
import subprocess
import sys
from collections import Counter, defaultdict
from pathlib import Path

if importlib.util.find_spec('cv2') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python-headless'])
if importlib.util.find_spec('yaml') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml'])

import cv2
import numpy as np
import yaml

SEED = 42
SPLIT_RATIOS = {'train': 0.80, 'valid': 0.10, 'test': 0.10}
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
FILENAME_GROUP_PATTERN = re.compile(
    r'^(?P<disease>Healthy|BG|WSSV_BG|WSSV)-(?P<shrimp_id>.+)-img-(?P<img_num>\d+)$',
    re.IGNORECASE,
)

# These thresholds were chosen for the currently supplied 1,452-image export.
# Candidate hashes are only a fast filter; the MAE + global SSIM check decides
# whether two images form the same visual duplicate component.
PHASH_MAX_DISTANCE = 8
AHASH_MAX_DISTANCE = 3
DHASH_MAX_DISTANCE = 4
MAX_NORMALIZED_MAE = 0.0065
MIN_GLOBAL_SSIM = 0.985
VERIFY_RESOLUTION = 256

# Set to True only to discard and recreate the generated dataset below. This
# target is derived data under WORK_ROOT, never the Roboflow download itself.
REBUILD_PREPARED_SPLIT = False
PREPARED_DIR = WORK_ROOT / 'mr_tu_shrimpdiseaseseg_dat_copy_v1_yolo26_grouped_near_duplicate_split'


def find_dataset_root(location: Path) -> Path:
    location = Path(location)
    if (location / 'data.yaml').is_file():
        return location
    candidates = sorted(location.rglob('data.yaml'))
    if not candidates:
        raise FileNotFoundError(f'No data.yaml found below {location}')
    return candidates[0].parent


def normalize_roboflow_stem(stem: str) -> str:
    stem = re.sub(r'_(jpg|jpeg|png|bmp|webp)\.rf\.[0-9a-f]+$', '', stem, flags=re.IGNORECASE)
    return re.sub(r'\.rf\.[0-9a-f]+$', '', stem, flags=re.IGNORECASE)


def filename_group_key(image_name: str) -> tuple[str | None, str | None]:
    stem = normalize_roboflow_stem(Path(image_name).stem)
    match = FILENAME_GROUP_PATTERN.match(stem)
    if not match:
        return None, None
    disease = match.group('disease').lower()
    shrimp_id = match.group('shrimp_id')
    return f'filename::{disease}::{shrimp_id}', f'{disease}::{shrimp_id}'


def label_path_for(image_path: Path) -> Path:
    labels_dir = image_path.parent.parent / 'labels'
    direct = labels_dir / f'{image_path.stem}.txt'
    if direct.exists():
        return direct
    matches = sorted(labels_dir.glob(f'{image_path.stem}*.txt'))
    return matches[0] if matches else direct


def prepared_image_name(source_name: str) -> str:
    # Roboflow exports can contain very long scraped-web filenames. A compact,
    # deterministic destination name keeps the derived data usable on Windows,
    # Kaggle and Colab while the manifest preserves the original filename.
    suffix = Path(source_name).suffix.lower()
    digest = hashlib.sha256(source_name.encode('utf-8')).hexdigest()[:20]
    return f'img_{digest}{suffix}'


def parse_class_names(raw_yaml: dict) -> dict[int, str]:
    names = raw_yaml.get('names', {})
    if isinstance(names, list):
        return {index: str(value) for index, value in enumerate(names)}
    return {int(key): str(value) for key, value in names.items()}


def label_ids(label_path: Path, image_name: str, class_names: dict[int, str]) -> tuple[int, ...]:
    if not label_path.exists():
        return ()
    ids = set()
    for line_number, line in enumerate(label_path.read_text(encoding='utf-8').splitlines(), start=1):
        tokens = line.strip().split()
        if not tokens:
            continue
        if len(tokens) < 7 or len(tokens[1:]) % 2 != 0:
            raise ValueError(f'Malformed YOLO segmentation polygon: {image_name}:{line_number}')
        try:
            class_id = int(float(tokens[0]))
            coordinates = [float(value) for value in tokens[1:]]
        except ValueError as error:
            raise ValueError(f'Non-numeric YOLO segmentation label: {image_name}:{line_number}') from error
        if class_id not in class_names:
            raise ValueError(f'Class id {class_id} is absent from data.yaml: {image_name}:{line_number}')
        if any(value < 0.0 or value > 1.0 for value in coordinates):
            raise ValueError(f'Polygon coordinate outside [0, 1]: {image_name}:{line_number}')
        ids.add(class_id)
    return tuple(sorted(ids))


def bits_to_int(bits: np.ndarray) -> int:
    value = 0
    for bit in bits.ravel():
        value = (value << 1) | int(bool(bit))
    return value


def image_hashes(image_path: Path) -> tuple[int, int, int]:
    image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f'Unreadable image: {image_path}')
    a_small = cv2.resize(image, (8, 8), interpolation=cv2.INTER_AREA)
    a_hash = bits_to_int(a_small >= a_small.mean())
    d_small = cv2.resize(image, (9, 8), interpolation=cv2.INTER_AREA)
    d_hash = bits_to_int(d_small[:, 1:] >= d_small[:, :-1])
    p_small = cv2.resize(image, (32, 32), interpolation=cv2.INTER_AREA).astype(np.float32)
    dct = cv2.dct(p_small)[:8, :8]
    median = np.median(dct.ravel()[1:])
    p_hash = bits_to_int(dct >= median)
    return p_hash, a_hash, d_hash


def hamming_distance(left: int, right: int) -> int:
    return (left ^ right).bit_count()


def normalized_image(image_path: Path) -> np.ndarray:
    image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f'Unreadable image: {image_path}')
    return cv2.resize(image, (VERIFY_RESOLUTION, VERIFY_RESOLUTION), interpolation=cv2.INTER_AREA).astype(np.float32) / 255.0


def global_ssim(left: np.ndarray, right: np.ndarray) -> float:
    mean_left, mean_right = float(left.mean()), float(right.mean())
    variance_left, variance_right = float(left.var()), float(right.var())
    covariance = float(((left - mean_left) * (right - mean_right)).mean())
    c1, c2 = 0.01 ** 2, 0.03 ** 2
    return ((2 * mean_left * mean_right + c1) * (2 * covariance + c2)) / (
        (mean_left ** 2 + mean_right ** 2 + c1) * (variance_left + variance_right + c2)
    )


class UnionFind:
    def __init__(self, values):
        self.parent = {value: value for value in values}

    def find(self, value):
        parent = self.parent[value]
        if parent != value:
            self.parent[value] = self.find(parent)
        return self.parent[value]

    def union(self, left, right):
        left_root, right_root = self.find(left), self.find(right)
        if left_root != right_root:
            self.parent[right_root] = left_root


def verified_near_duplicate_edges(records: list[dict]) -> list[dict]:
    cached_normalized = {}
    for record in records:
        record['p_hash'], record['a_hash'], record['d_hash'] = image_hashes(record['image_path'])

    edges = []
    for left_index, left in enumerate(records):
        for right in records[left_index + 1:]:
            p_distance = hamming_distance(left['p_hash'], right['p_hash'])
            a_distance = hamming_distance(left['a_hash'], right['a_hash'])
            d_distance = hamming_distance(left['d_hash'], right['d_hash'])
            if not (
                p_distance <= PHASH_MAX_DISTANCE
                and a_distance <= AHASH_MAX_DISTANCE
                and d_distance <= DHASH_MAX_DISTANCE
            ):
                continue
            left_image = cached_normalized.setdefault(left['name'], normalized_image(left['image_path']))
            right_image = cached_normalized.setdefault(right['name'], normalized_image(right['image_path']))
            mae = float(np.abs(left_image - right_image).mean())
            ssim = global_ssim(left_image, right_image)
            if mae <= MAX_NORMALIZED_MAE and ssim >= MIN_GLOBAL_SSIM:
                edges.append({
                    'left': left['name'],
                    'right': right['name'],
                    'p_hash_distance': p_distance,
                    'a_hash_distance': a_distance,
                    'd_hash_distance': d_distance,
                    'normalized_mae': round(mae, 7),
                    'global_ssim': round(ssim, 7),
                    'label_conflict': left['label_ids'] != right['label_ids'],
                })
    return edges


def target_group_counts(number_of_groups: int) -> dict[str, int]:
    if number_of_groups == 0:
        return {split: 0 for split in SPLIT_RATIOS}
    if number_of_groups == 1:
        return {'train': 1, 'valid': 0, 'test': 0}
    if number_of_groups == 2:
        return {'train': 1, 'valid': 0, 'test': 1}
    valid = max(1, int(round(number_of_groups * SPLIT_RATIOS['valid'])))
    test = max(1, int(round(number_of_groups * SPLIT_RATIOS['test'])))
    train = number_of_groups - valid - test
    if train < 1:
        train = 1
        if valid >= test:
            valid -= 1
        else:
            test -= 1
    return {'train': train, 'valid': valid, 'test': test}


def split_grouped_records(groups: list[dict], seed: int) -> dict[str, str]:
    by_stratum = defaultdict(list)
    for group in groups:
        by_stratum[group['label_stratum']].append(group)

    assignments = {}
    for stratum_index, (stratum, stratum_groups) in enumerate(sorted(by_stratum.items())):
        shuffled = sorted(stratum_groups, key=lambda item: item['group_id'])
        random.Random(seed + stratum_index).shuffle(shuffled)
        group_targets = target_group_counts(len(shuffled))
        image_targets = {
            split: sum(len(group['members']) for group in shuffled) * SPLIT_RATIOS[split]
            for split in SPLIT_RATIOS
        }
        group_counts = Counter()
        image_counts = Counter()
        for group in sorted(shuffled, key=lambda item: (-len(item['members']), item['group_id'])):
            available = [split for split in SPLIT_RATIOS if group_counts[split] < group_targets[split]]
            if not available:
                available = list(SPLIT_RATIOS)
            split = min(
                available,
                key=lambda candidate: (
                    image_counts[candidate] / max(image_targets[candidate], 1.0),
                    group_counts[candidate] / max(group_targets[candidate], 1),
                    candidate,
                ),
            )
            group_counts[split] += 1
            image_counts[split] += len(group['members'])
            for record in group['members']:
                assignments[record['name']] = split
        print(
            f'{stratum}: groups={len(shuffled)} -> '
            + ', '.join(f'{split}={group_counts[split]} groups/{image_counts[split]} images' for split in SPLIT_RATIOS)
        )
    return assignments


def link_or_copy(source: Path, destination: Path):
    destination.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.link(source, destination)
    except OSError:
        shutil.copy2(source, destination)


def safe_reset_prepared_dir(path: Path):
    if path.resolve().parent != WORK_ROOT.resolve():
        raise RuntimeError(f'Refusing to reset a directory outside WORK_ROOT: {path}')
    if path.name != 'mr_tu_shrimpdiseaseseg_dat_copy_v1_yolo26_grouped_near_duplicate_split':
        raise RuntimeError(f'Refusing to reset an unexpected directory: {path}')
    shutil.rmtree(path)


RAW_DATASET_DIR = find_dataset_root(Path(DATASET_LOCATION))
raw_yaml_path = RAW_DATASET_DIR / 'data.yaml'
raw_yaml = yaml.safe_load(raw_yaml_path.read_text(encoding='utf-8')) or {}
CLASS_NAMES = parse_class_names(raw_yaml)
if not CLASS_NAMES:
    raise ValueError('data.yaml contains no class names.')

def raw_dataset_fingerprint(dataset_root: Path) -> tuple[str, int]:
    digest = hashlib.sha256()
    digest.update(raw_yaml_path.read_bytes())
    image_count = 0
    for split_name in ('train', 'valid', 'test'):
        image_dir = dataset_root / split_name / 'images'
        if not image_dir.exists():
            continue
        for image_path in sorted(path for path in image_dir.iterdir() if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS):
            label_path = label_path_for(image_path)
            label_digest = hashlib.sha256(label_path.read_bytes()).hexdigest() if label_path.exists() else 'missing'
            digest.update(f'{split_name}/{image_path.name}|{image_path.stat().st_size}|{label_digest}\n'.encode('utf-8'))
            image_count += 1
    return digest.hexdigest(), image_count


SOURCE_FINGERPRINT, SOURCE_IMAGE_COUNT = raw_dataset_fingerprint(RAW_DATASET_DIR)
expected_image_count = globals().get('EXPECTED_IMAGE_COUNT')
if expected_image_count is not None and SOURCE_IMAGE_COUNT != expected_image_count:
    raise RuntimeError(
        f'Wrong downloaded export: expected {expected_image_count} images, found {SOURCE_IMAGE_COUNT}. '
        'Stop and verify the Roboflow project/version before splitting.'
    )

existing_manifest = PREPARED_DIR / 'split_manifest.csv'
existing_yaml = PREPARED_DIR / 'data.yaml'
existing_audit = PREPARED_DIR / 'split_audit.json'
if PREPARED_DIR.exists() and not REBUILD_PREPARED_SPLIT:
    if not (existing_manifest.is_file() and existing_yaml.is_file() and existing_audit.is_file()):
        raise RuntimeError(
            f'{PREPARED_DIR} already exists but is incomplete. Set REBUILD_PREPARED_SPLIT=True to recreate it.'
        )
    prior_audit = json.loads(existing_audit.read_text(encoding='utf-8'))
    if prior_audit.get('source_fingerprint') != SOURCE_FINGERPRINT:
        raise RuntimeError(
            'The existing prepared split belongs to a different raw export and will not be reused. '
            'Set REBUILD_PREPARED_SPLIT=True to recreate this derived dataset.'
        )
    base_path = PREPARED_DIR
    data_yaml_path = existing_yaml
    print('Reusing prepared grouped split for the verified current export:', PREPARED_DIR)
    print('Manifest:', existing_manifest)
else:
    if PREPARED_DIR.exists():
        safe_reset_prepared_dir(PREPARED_DIR)

    records = []
    seen_names = set()
    for original_split in ('train', 'valid', 'test'):
        image_dir = RAW_DATASET_DIR / original_split / 'images'
        if not image_dir.exists():
            continue
        for image_path in sorted(image_dir.iterdir()):
            if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue
            if image_path.name in seen_names:
                raise RuntimeError(f'Duplicate image filename across source splits: {image_path.name}')
            seen_names.add(image_path.name)
            label_path = label_path_for(image_path)
            group_key, parsed_group = filename_group_key(image_path.name)
            records.append({
                'name': image_path.name,
                'image_path': image_path,
                'label_path': label_path,
                'prepared_name': prepared_image_name(image_path.name),
                'original_split': original_split,
                'filename_group': parsed_group or '',
                'initial_group': group_key or f'singleton::{image_path.name}',
                'label_ids': label_ids(label_path, image_path.name, CLASS_NAMES),
            })

    if not records:
        raise RuntimeError(f'No supported images found below {RAW_DATASET_DIR}')
    print(f'Loaded {len(records)} images from raw export: {RAW_DATASET_DIR}')

    duplicate_edges = verified_near_duplicate_edges(records)
    print(f'Verified near-duplicate edges: {len(duplicate_edges)}')

    union_find = UnionFind(record['name'] for record in records)
    first_by_filename_group = {}
    for record in records:
        initial_group = record['initial_group']
        if initial_group in first_by_filename_group:
            union_find.union(first_by_filename_group[initial_group], record['name'])
        else:
            first_by_filename_group[initial_group] = record['name']
    for edge in duplicate_edges:
        union_find.union(edge['left'], edge['right'])

    by_component = defaultdict(list)
    for record in records:
        by_component[union_find.find(record['name'])].append(record)

    grouped_records = []
    for component_root, members in sorted(by_component.items()):
        members = sorted(members, key=lambda item: item['name'])
        unioned_label_ids = tuple(sorted({class_id for item in members for class_id in item['label_ids']}))
        label_stratum = '+'.join(map(str, unioned_label_ids)) if unioned_label_ids else 'healthy_empty'
        label_signatures = {item['label_ids'] for item in members}
        grouped_records.append({
            'group_id': f'component::{members[0]["name"]}',
            'component_root': component_root,
            'members': members,
            'label_stratum': label_stratum,
            'label_conflict': len(label_signatures) > 1,
        })

    assignments = split_grouped_records(grouped_records, SEED)
    if len(assignments) != len(records):
        raise RuntimeError('Not every image received a split assignment.')

    PREPARED_DIR.mkdir(parents=True, exist_ok=True)
    group_for_name = {
        record['name']: group
        for group in grouped_records
        for record in group['members']
    }
    rows = []
    for record in records:
        split = assignments[record['name']]
        image_destination = PREPARED_DIR / split / 'images' / record['prepared_name']
        label_destination = PREPARED_DIR / split / 'labels' / f'{Path(record["prepared_name"]).stem}.txt'
        link_or_copy(record['image_path'], image_destination)
        if record['label_path'].exists():
            link_or_copy(record['label_path'], label_destination)
        else:
            label_destination.parent.mkdir(parents=True, exist_ok=True)
            label_destination.write_text('', encoding='utf-8')
        group = group_for_name[record['name']]
        rows.append({
            'image_name': record['name'],
            'prepared_image_name': record['prepared_name'],
            'source_image': str(record['image_path']),
            'source_label': str(record['label_path']),
            'source_split': record['original_split'],
            'filename_group': record['filename_group'],
            'component_id': group['group_id'],
            'label_stratum': group['label_stratum'],
            'image_label_ids': '+'.join(map(str, record['label_ids'])) or 'healthy_empty',
            'component_label_conflict': group['label_conflict'],
            'split': split,
        })

    with (PREPARED_DIR / 'split_manifest.csv').open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(sorted(rows, key=lambda item: item['image_name']))
    with (PREPARED_DIR / 'near_duplicate_edges.json').open('w', encoding='utf-8') as handle:
        json.dump(duplicate_edges, handle, indent=2, ensure_ascii=False)

    split_image_counts = Counter(assignments.values())
    conflict_groups = [group for group in grouped_records if group['label_conflict']]
    audit = {
        'seed': SEED,
        'ratios': SPLIT_RATIOS,
        'raw_dataset': str(RAW_DATASET_DIR),
        'expected_project_id': globals().get('EXPECTED_PROJECT_ID'),
        'source_fingerprint': SOURCE_FINGERPRINT,
        'source_image_count': SOURCE_IMAGE_COUNT,
        'images': len(records),
        'groups': len(grouped_records),
        'filename_grouped_images': sum(bool(record['filename_group']) for record in records),
        'unmatched_filename_images': sum(not bool(record['filename_group']) for record in records),
        'verified_near_duplicate_edges': len(duplicate_edges),
        'label_conflict_groups': len(conflict_groups),
        'split_image_counts': dict(split_image_counts),
        'class_names': CLASS_NAMES,
    }
    (PREPARED_DIR / 'split_audit.json').write_text(json.dumps(audit, indent=2, ensure_ascii=False), encoding='utf-8')

    prepared_yaml = dict(raw_yaml)
    prepared_yaml['train'] = str((PREPARED_DIR / 'train' / 'images').resolve())
    prepared_yaml['val'] = str((PREPARED_DIR / 'valid' / 'images').resolve())
    prepared_yaml['test'] = str((PREPARED_DIR / 'test' / 'images').resolve())
    prepared_yaml['nc'] = len(CLASS_NAMES)
    prepared_yaml['names'] = [CLASS_NAMES[index] for index in sorted(CLASS_NAMES)]
    prepared_yaml['split_policy'] = 'filename groups + verified visual near-duplicate components; group-stratified 80/10/10'
    data_yaml_path = PREPARED_DIR / 'data.yaml'
    data_yaml_path.write_text(yaml.safe_dump(prepared_yaml, sort_keys=False, allow_unicode=True), encoding='utf-8')

    base_path = PREPARED_DIR
    print('Prepared split:', PREPARED_DIR)
    print('Image counts:', dict(sorted(split_image_counts.items())))
    print('Groups with heterogeneous labels (review in split_manifest.csv):', len(conflict_groups))
    print('Audit:', PREPARED_DIR / 'split_audit.json')

base_path = Path(base_path)
data_yaml_path = Path(data_yaml_path)
print('Prepared data.yaml:', data_yaml_path)


In [ ]:
from pathlib import Path

if "WORK_ROOT" not in globals():
    WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else (Path("/content") if Path("/content").exists() else Path.cwd())
import yaml

if "base_path" not in globals() or "data_yaml_path" not in globals():
    raise RuntimeError("Run the grouped near-duplicate split cell before this validation cell.")
if "data_yaml_path" not in globals():
    data_yaml_path = str(Path(base_path) / "data.yaml")

config_path = Path(data_yaml_path)
if not config_path.is_file():
    raise FileNotFoundError(f"Prepared data.yaml not found: {config_path}")
config = yaml.safe_load(config_path.read_text(encoding="utf-8"))

for split_key, folder in (("train", "train"), ("val", "valid"), ("test", "test")):
    image_dir = Path(base_path) / folder / "images"
    label_dir = Path(base_path) / folder / "labels"
    images = [path for path in image_dir.iterdir() if path.is_file()]
    labels = list(label_dir.glob("*.txt"))
    if not images:
        raise RuntimeError(f"{split_key} split is empty.")
    if len(images) != len(labels):
        raise RuntimeError(
            f"{split_key} image/label mismatch: {len(images)} images, {len(labels)} labels."
        )
    print(f"{split_key}: {len(images)} images, {len(labels)} labels")
print("Verified names:", config.get("names"))
print("Using:", config_path)


In [ ]:
# Runtime patch architecture modules into Ultralytics namespace.
# This avoids fragile source-file text patches and keeps YAML parsing safe.
import torch
import torch.nn as nn

import ultralytics
import ultralytics.nn.modules as nn_modules
import ultralytics.nn.modules.conv as conv_module
import ultralytics.nn.tasks as tasks_module


class SimAM(nn.Module):
    def __init__(self, c1=None, e_lambda=1e-4):
        super().__init__()
        self.e_lambda = e_lambda
        self.activation = nn.Sigmoid()

    def forward(self, x):
        b, c, h, w = x.size()
        n = h * w - 1
        if n <= 0:
            return x
        x_mu = x - x.mean(dim=[2, 3], keepdim=True)
        denom = 4 * (x_mu.pow(2).sum(dim=[2, 3], keepdim=True) / n + self.e_lambda)
        y = x_mu.pow(2) / denom + 0.5
        return x * self.activation(y)


class CoordAtt(nn.Module):
    """Channel-preserving Coordinate Attention.

    Compatible with both clean parse_model calls (CoordAtt(c1, reduction))
    and older patched parse_model calls (CoordAtt(c1, c2, reduction)).
    """

    def __init__(self, c1, c2=None, reduction=32):
        super().__init__()
        if c2 is not None and c2 != c1 and reduction == 32:
            # Clean parse_model passes YAML args directly: [c1, reduction].
            reduction = c2
            c2 = c1
        c2 = c1 if c2 is None else c2
        mip = max(8, c1 // reduction)
        self.conv1 = nn.Conv2d(c1, mip, 1, 1, 0)
        self.bn1 = nn.BatchNorm2d(mip)
        self.act = nn.SiLU()
        self.conv_h = nn.Conv2d(mip, c2, 1, 1, 0)
        self.conv_w = nn.Conv2d(mip, c2, 1, 1, 0)
        self.proj = nn.Identity() if c1 == c2 else nn.Conv2d(c1, c2, 1, 1, 0)

    def forward(self, x):
        identity = self.proj(x)
        n, c, h, w = x.size()
        x_h = x.mean(dim=3, keepdim=True)
        x_w = x.mean(dim=2, keepdim=True).permute(0, 1, 3, 2)
        y = self.act(self.bn1(self.conv1(torch.cat([x_h, x_w], dim=2))))
        x_h, x_w = torch.split(y, [h, w], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)
        return identity * self.conv_h(x_h).sigmoid() * self.conv_w(x_w).sigmoid()


class LargeKernelAttention(nn.Module):
    """Channel-preserving LKA. YAML args: [actual_channels]."""

    def __init__(self, c1, kernel_size=5, dilation_kernel_size=7, dilation=3):
        super().__init__()
        self.dw = nn.Conv2d(c1, c1, kernel_size, padding=kernel_size // 2, groups=c1)
        padding = dilation * (dilation_kernel_size - 1) // 2
        self.dw_d = nn.Conv2d(c1, c1, dilation_kernel_size, padding=padding, dilation=dilation, groups=c1)
        self.pw = nn.Conv2d(c1, c1, 1)
        self.gate = nn.Sigmoid()

    def forward(self, x):
        return x * self.gate(self.pw(self.dw_d(self.dw(x))))


for namespace in (conv_module, nn_modules, tasks_module):
    namespace.SimAM = SimAM
    namespace.CoordAtt = CoordAtt
    namespace.LargeKernelAttention = LargeKernelAttention

existing_all = list(getattr(nn_modules, "__all__", []))
for name in ["SimAM", "CoordAtt", "LargeKernelAttention"]:
    if name not in existing_all:
        existing_all.append(name)
nn_modules.__all__ = existing_all

print("Ultralytics:", ultralytics.__version__)
print("Registered modules:", SimAM.__name__, CoordAtt.__name__, LargeKernelAttention.__name__)


In [ ]:
# Write architecture YAML and smoke-build the model.
# Important channel note for YOLO11n:
# layer 16/18 P3 = 64 channels, layer 19/21 P4 = 128 channels, layer 22/24 P5 = 256 channels after width scaling.
from pathlib import Path

from ultralytics import YOLO

EXPERIMENT_KEY = "lka_simam_head_strong_mrtu_grouped_near_duplicate_split"
EXPERIMENT_NAME = "YOLO11n-seg + LKA->SimAM head (strong, new data)"
YAML_FILENAME = "yolo11n-seg-lka-simam-head.yaml"
YAML_CONTENT = r"""# YOLO11n-seg + LargeKernelAttention->SimAM on P3/P4/P5 head outputs
nc: 2
scales:
  n: [0.50, 0.25, 1024]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 2, C3k2, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]
  - [-1, 2, C2PSA, [1024]]

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, False]]

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 2, C3k2, [1024, True]]

  - [16, 1, LargeKernelAttention, [64]]
  - [23, 1, SimAM, []]
  - [19, 1, LargeKernelAttention, [128]]
  - [25, 1, SimAM, []]
  - [22, 1, LargeKernelAttention, [256]]
  - [27, 1, SimAM, []]
  - [[24, 26, 28], 1, Segment, [nc, 32, 256]]"""

EXPERIMENT_ROOT = WORK_ROOT / "yolov11n_simam_NhomC" / EXPERIMENT_KEY
CFG_DIR = EXPERIMENT_ROOT / "cfg"
CFG_DIR.mkdir(parents=True, exist_ok=True)
yaml_path = CFG_DIR / YAML_FILENAME
yaml_path.write_text(YAML_CONTENT)
print("YAML written to:", yaml_path)

# Smoke build catches channel, concat-size, and custom-module registration errors before training.
smoke_model = YOLO(str(yaml_path))
smoke_model.info()
print("Smoke build OK:", EXPERIMENT_NAME)


In [ ]:
# Evaluation helpers reused from the clean baseline style
import gc
import shutil
import time
from pathlib import Path

import pandas as pd
import yaml
from ultralytics import YOLO

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

COUNT_PENALTY_WEIGHT = 0.05
DISEASE_MISS_PENALTY_WEIGHT = 0.15
HEALTHY_FP_PENALTY_WEIGHT = 0.10
PREDICT_CONF_FOR_COUNT = 0.25

STRONG_AUG_TRAIN_ARGS = {
    "auto_augment": None,
    "erasing": 0.15,
    "mosaic": 0.0,
    "mixup": 0.0,
    "cutmix": 0.0,
    "copy_paste": 0.0,
    "fliplr": 0.5,
    "flipud": 0.3,
    "hsv_h": 0.05,
    "hsv_s": 0.50,
    "hsv_v": 0.40,
    "degrees": 10.0,
    "translate": 0.10,
    "scale": 0.50,
    "shear": 0.0,
    "perspective": 0.0,
    "multi_scale": 0.0,
    "bgr": 0.0,
}


def disable_ultralytics_albumentations():
    try:
        import ultralytics.data.augment as yolo_augment
    except Exception as exc:
        print(f"Could not patch Ultralytics Albumentations hook: {exc}")
        return

    class NoOpAlbumentations:
        contains_spatial = False

        def __init__(self, *args, **kwargs):
            self.transform = None

        def __call__(self, labels):
            return labels

    yolo_augment.Albumentations = NoOpAlbumentations
    print("Ultralytics Albumentations hook disabled.")


def remove_yolo_label_caches(root):
    for cache_path in Path(root).glob("**/*.cache"):
        cache_path.unlink()


def find_image_for_label(image_dir, label_file):
    stem = Path(label_file).stem
    for ext in IMAGE_EXTENSIONS:
        candidate = Path(image_dir) / f"{stem}{ext}"
        if candidate.exists():
            return candidate
    return None


def write_data_yaml(dataset_dir, yaml_path, val_dir="valid", test_dir="test"):
    with open(data_yaml_path, "r") as f:
        content = yaml.safe_load(f)
    content["train"] = str(Path(dataset_dir) / "train" / "images")
    content["val"] = str(Path(dataset_dir) / val_dir / "images")
    content["test"] = str(Path(dataset_dir) / test_dir / "images")
    with open(yaml_path, "w") as f:
        yaml.safe_dump(content, f, sort_keys=False)
    return yaml_path


def copy_split_by_label_state(src_dataset, dst_dataset, split, want_labeled):
    src_images = Path(src_dataset) / split / "images"
    src_labels = Path(src_dataset) / split / "labels"
    dst_images = Path(dst_dataset) / split / "images"
    dst_labels = Path(dst_dataset) / split / "labels"
    dst_images.mkdir(parents=True, exist_ok=True)
    dst_labels.mkdir(parents=True, exist_ok=True)
    copied = 0
    for label_path in sorted(src_labels.glob("*.txt")):
        lines = [line.strip() for line in label_path.read_text().splitlines() if line.strip()]
        if bool(lines) != want_labeled:
            continue
        image_path = find_image_for_label(src_images, label_path.name)
        if image_path is None:
            continue
        shutil.copy2(image_path, dst_images / image_path.name)
        shutil.copy2(label_path, dst_labels / label_path.name)
        copied += 1
    return copied


def make_state_eval_dataset(src_dataset, exp_key, state_name, want_labeled):
    dst = EXPERIMENT_ROOT / f"dataset_{state_name}_eval"
    if dst.exists():
        shutil.rmtree(dst)
    for sub in ["images", "labels"]:
        (dst / "train" / sub).mkdir(parents=True, exist_ok=True)
    copied = {}
    for split in ["valid", "test"]:
        copied[split] = copy_split_by_label_state(src_dataset, dst, split, want_labeled=want_labeled)
    yaml_path = dst / f"data_{state_name}.yaml"
    write_data_yaml(dst, yaml_path)
    print(f"{state_name} eval dataset for {exp_key}: {copied}")
    return dst, yaml_path, copied


def metric_value(metrics, dotted_path, default=float("nan")):
    obj = metrics
    for part in dotted_path.split("."):
        if not hasattr(obj, part):
            return default
        obj = getattr(obj, part)
    try:
        return float(obj)
    except Exception:
        return default


def list_images(images_dir):
    image_paths = []
    for ext in IMAGE_EXTENSIONS:
        image_paths.extend(Path(images_dir).glob(f"*{ext}"))
    return sorted(image_paths)


def count_prediction_errors(model, images_dir, labels_dir, conf=PREDICT_CONF_FOR_COUNT):
    image_paths = list_images(images_dir)
    results = model.predict(source=[str(p) for p in image_paths], imgsz=640, conf=conf, verbose=False)
    box_errors, mask_errors, box_exact, mask_exact = [], [], [], []
    gt_total = pred_box_total = pred_mask_total = disease_images = 0
    disease_box_miss_images = disease_mask_miss_images = 0

    for image_path, result in zip(image_paths, results):
        label_path = Path(labels_dir) / f"{image_path.stem}.txt"
        gt_count = 0
        if label_path.exists():
            gt_count = len([line for line in label_path.read_text().splitlines() if line.strip()])
        box_count = len(result.boxes) if result.boxes is not None else 0
        mask_count = len(result.masks) if result.masks is not None else 0
        denom = max(1, gt_count)
        box_errors.append(abs(box_count - gt_count) / denom)
        mask_errors.append(abs(mask_count - gt_count) / denom)
        box_exact.append(float(box_count == gt_count))
        mask_exact.append(float(mask_count == gt_count))
        if gt_count > 0:
            disease_images += 1
            disease_box_miss_images += int(box_count == 0)
            disease_mask_miss_images += int(mask_count == 0)
        gt_total += gt_count
        pred_box_total += box_count
        pred_mask_total += mask_count

    return {
        "images": len(image_paths),
        "gt_total": gt_total,
        "pred_box_total": pred_box_total,
        "pred_mask_total": pred_mask_total,
        "box_count_mae": sum(box_errors) / len(box_errors) if box_errors else float("nan"),
        "mask_count_mae": sum(mask_errors) / len(mask_errors) if mask_errors else float("nan"),
        "box_count_exact": sum(box_exact) / len(box_exact) if box_exact else float("nan"),
        "mask_count_exact": sum(mask_exact) / len(mask_exact) if mask_exact else float("nan"),
        "disease_images": disease_images,
        "disease_box_miss_images": disease_box_miss_images,
        "disease_mask_miss_images": disease_mask_miss_images,
        "disease_box_miss_rate": disease_box_miss_images / disease_images if disease_images else float("nan"),
        "disease_mask_miss_rate": disease_mask_miss_images / disease_images if disease_images else float("nan"),
    }


def healthy_false_positive_summary(model, images_dir, conf=PREDICT_CONF_FOR_COUNT):
    image_paths = list_images(images_dir)
    if not image_paths:
        return {
            "healthy_images": 0,
            "healthy_mask_fp_rate": float("nan"),
            "healthy_box_fp_rate": float("nan"),
            "healthy_fp_masks_total": 0,
            "healthy_fp_masks_per_image": float("nan"),
            "healthy_avg_fp_confidence": float("nan"),
        }
    results = model.predict(source=[str(p) for p in image_paths], imgsz=640, conf=conf, verbose=False)
    images_with_box_fp = images_with_mask_fp = box_total = mask_total = 0
    confidences = []
    for result in results:
        box_count = len(result.boxes) if result.boxes is not None else 0
        mask_count = len(result.masks) if result.masks is not None else 0
        if box_count > 0:
            images_with_box_fp += 1
            try:
                confidences.extend([float(v) for v in result.boxes.conf.detach().cpu().tolist()])
            except Exception:
                pass
        if mask_count > 0:
            images_with_mask_fp += 1
        box_total += box_count
        mask_total += mask_count
    n = len(image_paths)
    return {
        "healthy_images": n,
        "healthy_mask_fp_rate": images_with_mask_fp / n,
        "healthy_box_fp_rate": images_with_box_fp / n,
        "healthy_fp_masks_total": mask_total,
        "healthy_fp_masks_per_image": mask_total / n,
        "healthy_avg_fp_confidence": sum(confidences) / len(confidences) if confidences else 0.0,
    }


def read_best_epoch_from_results(run_path):
    results_csv = Path(run_path) / "results.csv"
    if not results_csv.exists():
        return {}
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    mask_col = "metrics/mAP50(M)"
    if mask_col not in df.columns:
        return {"epochs_ran": len(df)}
    best_idx = df[mask_col].idxmax()
    best = df.iloc[best_idx]
    last = df.iloc[-1]
    return {
        "epochs_ran": int(len(df)),
        "best_epoch_by_mask_map50": int(best["epoch"]) if "epoch" in df.columns else int(best_idx + 1),
        "best_val_mask_map50": float(best.get(mask_col, float("nan"))),
        "best_val_mask_map50_95": float(best.get("metrics/mAP50-95(M)", float("nan"))),
        "last_train_seg_loss": float(last.get("train/seg_loss", float("nan"))),
        "last_val_seg_loss": float(last.get("val/seg_loss", float("nan"))),
        "seg_loss_gap_val_minus_train": float(last.get("val/seg_loss", float("nan")) - last.get("train/seg_loss", float("nan"))),
    }


### Required preflight: paths, segmentation labels, and model shapes

This gate runs immediately before any training. It stops the notebook if a split path is invalid, an image has no matching label file, a polygon label is malformed, or the configured model cannot complete a dummy segmentation forward pass.


In [ ]:
# Mandatory preflight gate. Do not bypass this cell before a real training run.
from pathlib import Path
import math

import torch
import yaml
from ultralytics import YOLO

PREFLIGHT_IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def _preflight_image_dir(value, data_config_path):
    """Resolve a YOLO data.yaml split value to its images directory."""
    if isinstance(value, (list, tuple)):
        if len(value) != 1:
            raise ValueError("This notebook requires exactly one image directory per split.")
        value = value[0]
    image_dir = Path(str(value))
    if not image_dir.is_absolute():
        image_dir = data_config_path.parent / image_dir
    return image_dir.resolve()


def _preflight_validate_dataset(data_config_path):
    data_config_path = Path(data_config_path).resolve()
    if not data_config_path.is_file():
        raise FileNotFoundError(f"Prepared data.yaml was not found: {data_config_path}")

    config = yaml.safe_load(data_config_path.read_text(encoding="utf-8")) or {}
    names = config.get("names", {})
    class_count = len(names) if isinstance(names, (list, tuple, dict)) else 0
    if class_count < 1:
        raise ValueError("data.yaml must define at least one class in `names`.")

    summary = {}
    for split_name in ("train", "val", "test"):
        if split_name not in config:
            raise KeyError(f"data.yaml is missing the `{split_name}` split.")
        image_dir = _preflight_image_dir(config[split_name], data_config_path)
        label_dir = image_dir.parent / "labels"
        if not image_dir.is_dir() or not label_dir.is_dir():
            raise FileNotFoundError(
                f"{split_name}: expected images={image_dir} and labels={label_dir}"
            )

        images = sorted(
            path for path in image_dir.iterdir()
            if path.is_file() and path.suffix.lower() in PREFLIGHT_IMAGE_EXTENSIONS
        )
        if not images:
            raise RuntimeError(f"{split_name}: image split is empty: {image_dir}")

        label_files = sorted(path for path in label_dir.glob("*.txt") if path.is_file())
        image_stems = {path.stem for path in images}
        label_stems = {path.stem for path in label_files}
        missing_labels = sorted(image_stems - label_stems)
        orphan_labels = sorted(label_stems - image_stems)
        if missing_labels or orphan_labels:
            raise RuntimeError(
                f"{split_name}: image/label mismatch; "
                f"missing_labels={missing_labels[:5]}, orphan_labels={orphan_labels[:5]}"
            )

        labeled_images = 0
        polygons = 0
        for label_path in label_files:
            lines = [line.strip() for line in label_path.read_text(encoding="utf-8").splitlines() if line.strip()]
            if lines:
                labeled_images += 1
            for line_number, line in enumerate(lines, start=1):
                tokens = line.split()
                if len(tokens) < 7 or (len(tokens) - 1) % 2 != 0:
                    raise ValueError(
                        f"{split_name}: malformed segmentation polygon at "
                        f"{label_path.name}:{line_number}"
                    )
                try:
                    class_id = int(tokens[0])
                    coordinates = [float(token) for token in tokens[1:]]
                except ValueError as exc:
                    raise ValueError(
                        f"{split_name}: non-numeric label at {label_path.name}:{line_number}"
                    ) from exc
                if not 0 <= class_id < class_count:
                    raise ValueError(
                        f"{split_name}: class id {class_id} out of range at "
                        f"{label_path.name}:{line_number}"
                    )
                if any(not math.isfinite(value) or value < 0.0 or value > 1.0 for value in coordinates):
                    raise ValueError(
                        f"{split_name}: polygon coordinate outside [0, 1] at "
                        f"{label_path.name}:{line_number}"
                    )
                polygons += 1
        summary[split_name] = {
            "images": len(images),
            "empty_labels": len(images) - labeled_images,
            "polygons": polygons,
        }
    return summary


def _preflight_flatten_tensors(value):
    if torch.is_tensor(value):
        yield value
    elif isinstance(value, (tuple, list)):
        for item in value:
            yield from _preflight_flatten_tensors(item)
    elif isinstance(value, dict):
        for item in value.values():
            yield from _preflight_flatten_tensors(item)


def run_notebook_preflight(model_specs, expected_attention=None, dummy_imgsz=128):
    """Fail before training on a bad dataset path, label, model build, or tensor shape."""
    if dummy_imgsz % 32:
        raise ValueError("dummy_imgsz must be divisible by 32 for YOLO segmentation.")
    dataset_summary = _preflight_validate_dataset(data_yaml_path)
    print("Dataset preflight PASS:", dataset_summary)

    checked_specs = []
    for model_spec in dict.fromkeys(str(spec) for spec in model_specs):
        model = YOLO(model_spec)
        module = model.model.eval()
        attention_counts = {}
        attention_shape_errors = []
        attention_handles = []
        shape_preserving_names = {
            "CoTEBoundaryLiteGate", "DPCAGate", "LargeKernelAttention", "CoordAtt", "SimAM"
        }

        def _attention_shape_hook(layer_name):
            def hook(_module, inputs, output):
                input_tensor = inputs[0] if inputs else None
                if not torch.is_tensor(input_tensor) or not torch.is_tensor(output):
                    attention_shape_errors.append(f"{layer_name}: non-tensor attention input/output")
                elif input_tensor.ndim != 4 or output.ndim != 4 or tuple(input_tensor.shape) != tuple(output.shape):
                    attention_shape_errors.append(
                        f"{layer_name}: expected shape-preserving BCHW, got "
                        f"{tuple(input_tensor.shape)} -> {tuple(output.shape)}"
                    )
            return hook

        for layer in module.modules():
            layer_name = layer.__class__.__name__
            if layer_name in shape_preserving_names:
                attention_counts[layer_name] = attention_counts.get(layer_name, 0) + 1
                attention_handles.append(layer.register_forward_hook(_attention_shape_hook(layer_name)))
        try:
            device = next(module.parameters()).device
        except StopIteration:
            device = torch.device("cpu")
        dummy = torch.zeros(1, 3, dummy_imgsz, dummy_imgsz, device=device)
        with torch.inference_mode():
            output = module(dummy)
        for handle in attention_handles:
            handle.remove()
        if attention_shape_errors:
            raise RuntimeError(f"{model_spec}: attention shape check failed: {attention_shape_errors}")
        if expected_attention is not None:
            observed_attention = {name: attention_counts.get(name, 0) for name in expected_attention}
            if observed_attention != expected_attention:
                raise RuntimeError(
                    f"{model_spec}: attention module count mismatch; "
                    f"expected={expected_attention}, observed={observed_attention}"
                )
        tensors = list(_preflight_flatten_tensors(output))
        if not tensors:
            raise RuntimeError(f"{model_spec}: model forward returned no tensors.")
        if any(tensor.numel() == 0 or not torch.isfinite(tensor).all().item() for tensor in tensors):
            raise FloatingPointError(f"{model_spec}: model forward produced empty or non-finite tensors.")
        checked_specs.append(model_spec)
        del model, module, dummy, output, tensors
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print("Model build/shape preflight PASS:", checked_specs)
    return dataset_summary


In [ ]:

# Hard gate: do not start any training until all paths, polygons and model shapes pass.
run_notebook_preflight([str(yaml_path)], expected_attention={'LargeKernelAttention': 3, 'SimAM': 3})

# Training
from pathlib import Path

from ultralytics import YOLO

EXPERIMENT_ROOT = WORK_ROOT / "yolov11n_simam_NhomC" / EXPERIMENT_KEY
RUNS_DIR = WORK_ROOT / "runs" / "segment"
REPORT_DIR = EXPERIMENT_ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

dataset_dir = Path(base_path)
remove_yolo_label_caches(dataset_dir)
labeled_eval_dir, labeled_eval_yaml, _ = make_state_eval_dataset(dataset_dir, EXPERIMENT_KEY, "labeled_only", True)
healthy_eval_dir, healthy_eval_yaml, _ = make_state_eval_dataset(dataset_dir, EXPERIMENT_KEY, "healthy_only", False)

run_name = f"yolov11n_simam_{EXPERIMENT_KEY}"
disable_ultralytics_albumentations()

yolo = YOLO(str(yaml_path))
try:
    yolo.load("yolo11n-seg.pt")
    print("Loaded yolo11n-seg pretrained weights.")
except Exception as exc:
    print("Pretrained load warning:", exc)

start = time.time()
yolo.train(
    data=str(data_yaml_path),
    task="segment",
    imgsz=640,
    epochs=100,
    batch=16,
    patience=30,
    seed=42,
    deterministic=True,
    workers=0,
    project=str(RUNS_DIR),
    name=run_name,
    exist_ok=True,
    pretrained=True,
    plots=True,
    verbose=True,
    **STRONG_AUG_TRAIN_ARGS,
)
train_time_min = (time.time() - start) / 60

run_path = RUNS_DIR / run_name
best_model_path = run_path / "weights" / "best.pt"
print("Best checkpoint:", best_model_path)


In [ ]:
# Evaluation: full test, labeled-only test, and healthy-negative false positives
best_model = YOLO(str(best_model_path))

full_test = best_model.val(data=str(data_yaml_path), split="test", imgsz=640, plots=True, verbose=False)
labeled_test = best_model.val(data=str(labeled_eval_yaml), split="test", imgsz=640, plots=False, verbose=False)

test_count = count_prediction_errors(best_model, Path(base_path) / "test" / "images", Path(base_path) / "test" / "labels")
labeled_test_count = count_prediction_errors(best_model, labeled_eval_dir / "test" / "images", labeled_eval_dir / "test" / "labels")
healthy_test_fp = healthy_false_positive_summary(best_model, healthy_eval_dir / "test" / "images")

labeled_map50 = metric_value(labeled_test, "seg.map50")
healthy_aware_score = (
    labeled_map50
    - COUNT_PENALTY_WEIGHT * labeled_test_count["mask_count_mae"]
    - DISEASE_MISS_PENALTY_WEIGHT * labeled_test_count["disease_box_miss_rate"]
    - HEALTHY_FP_PENALTY_WEIGHT * healthy_test_fp["healthy_mask_fp_rate"]
)

row = {
    "experiment": EXPERIMENT_KEY,
    "name": EXPERIMENT_NAME,
    "run_name": run_name,
    "run_path": str(run_path),
    "best_pt": str(best_model_path),
    "train_time_min": round(train_time_min, 2),
    "full_test_box_map50": metric_value(full_test, "box.map50"),
    "full_test_mask_map50": metric_value(full_test, "seg.map50"),
    "full_test_mask_map50_95": metric_value(full_test, "seg.map"),
    "labeled_test_box_map50": metric_value(labeled_test, "box.map50"),
    "labeled_test_mask_map50": labeled_map50,
    "labeled_test_mask_map50_95": metric_value(labeled_test, "seg.map"),
    "test_gt_instances": test_count["gt_total"],
    "test_pred_masks": test_count["pred_mask_total"],
    "test_mask_count_mae": test_count["mask_count_mae"],
    "labeled_test_gt_instances": labeled_test_count["gt_total"],
    "labeled_test_pred_masks": labeled_test_count["pred_mask_total"],
    "labeled_test_mask_count_mae": labeled_test_count["mask_count_mae"],
    "labeled_test_disease_box_miss_rate": labeled_test_count["disease_box_miss_rate"],
    "labeled_test_disease_mask_miss_rate": labeled_test_count["disease_mask_miss_rate"],
    "healthy_test_images": healthy_test_fp["healthy_images"],
    "healthy_test_mask_fp_rate": healthy_test_fp["healthy_mask_fp_rate"],
    "healthy_test_box_fp_rate": healthy_test_fp["healthy_box_fp_rate"],
    "healthy_test_fp_masks_total": healthy_test_fp["healthy_fp_masks_total"],
    "healthy_test_fp_masks_per_image": healthy_test_fp["healthy_fp_masks_per_image"],
    "healthy_aware_labeled_test_mask_map50": healthy_aware_score,
}
row.update(read_best_epoch_from_results(run_path))

summary_df = pd.DataFrame([row])
summary_csv = REPORT_DIR / f"{EXPERIMENT_KEY}_summary.csv"
summary_df.to_csv(summary_csv, index=False)
display(summary_df)
print("Saved summary:", summary_csv)

del yolo, best_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
